# GemmaFit E2B Text-LoRA Adapter Eval

This Colab notebook evaluates the completed `gemmafit_v5_2_e2b_text_lora_tool_contract_v1` adapter without relying on `eval_loss`.

It repairs the common Colab `torchao` / `peft` incompatibility, verifies the adapter and checkpoints on Drive, regenerates the deterministic v5.2 dataset if needed, runs real model generation on held-out validation prompts, and scores JSON/tool/evidence/safety gates.

Default path: direct adapter eval. Optional path: merge fp16 adapter for later export by setting `MERGE_FP16=1` before running the optional merge cell.


In [ ]:
# Cell 0: Dependency repair. Run this first.
# If packages are upgraded, the cell restarts the runtime once. Then rerun from Cell 0.
from pathlib import Path
import importlib.metadata as metadata
import os
import re
import signal
import subprocess
import sys

SENTINEL = Path('/content/.gemmafit_text_lora_eval_deps_ready_20260511')

def version_tuple(value: str) -> tuple[int, ...]:
    nums = re.findall(r'\d+', value or '')
    return tuple(int(num) for num in nums[:4]) if nums else (0,)

def installed_version(package: str) -> str | None:
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

def pip_install(specs: list[str]) -> None:
    print('Installing:', ' '.join(specs))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *specs])

if os.environ.get('SKIP_DEP_REPAIR', '0') == '1':
    print('SKIP_DEP_REPAIR=1; dependency repair skipped.')
elif SENTINEL.exists():
    print('Dependency repair sentinel exists:', SENTINEL)
    for pkg in ['transformers', 'peft', 'accelerate', 'torchao', 'bitsandbytes']:
        print(pkg, installed_version(pkg))
else:
    specs = [
        'transformers>=4.51.0',
        'peft>=0.15.0',
        'accelerate',
        'safetensors',
        'bitsandbytes',
        'torchao>=0.16.0',
    ]
    try:
        pip_install(specs)
    except subprocess.CalledProcessError:
        print('torchao upgrade failed; falling back to uninstalling torchao so PEFT will not reject an old torchao build.')
        fallback_specs = [spec for spec in specs if not spec.startswith('torchao')]
        pip_install(fallback_specs)
        subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])

    SENTINEL.write_text('ok', encoding='utf-8')
    for pkg in ['transformers', 'peft', 'accelerate', 'torchao', 'bitsandbytes']:
        print(pkg, installed_version(pkg))
    if os.environ.get('RESTART_AFTER_DEP_REPAIR', '1') == '1':
        print('Dependencies repaired. Restarting runtime once; rerun all cells after reconnect.')
        os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
# Cell 1: Mount Drive, sync repo, and verify the completed text-LoRA adapter.
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import subprocess
import sys
from pathlib import Path

GITHUB_REPO = os.environ.get('GEMMAFIT_GITHUB_REPO', 'https://github.com/bkttt0429/GemmaFit.git')
GITHUB_REF = os.environ.get('GEMMAFIT_GITHUB_REF', 'codex/e2b-video-capability-probes')
REPO_DIR = Path(os.environ.get('GEMMAFIT_REPO_DIR', '/content/GemmaFit'))
DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
RUN_NAME = os.environ.get('RUN_NAME', 'google_gemma_4_E2B_it_gemmafit_v5_2_e2b_text_lora_tool_contract_v1')
BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', GITHUB_REF, '--depth', '1', GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', GITHUB_REF], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'checkout', GITHUB_REF], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', GITHUB_REF], cwd=str(REPO_DIR), check=True)

sys.path.insert(0, str(REPO_DIR / 'finetune'))

TRAINED = DRIVE_ROOT / 'trained_outputs'
CHECKPOINT_DIR = TRAINED / 'checkpoints' / RUN_NAME
ADAPTER_DIR = TRAINED / 'adapter' / RUN_NAME
MERGED_DIR = TRAINED / 'merged_fp16' / RUN_NAME
EVAL_DIR = TRAINED / 'evals'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

def sorted_checkpoints(path: Path) -> list[str]:
    items = []
    if path.exists():
        for checkpoint in path.glob('checkpoint-*'):
            try:
                step = int(checkpoint.name.split('-')[-1])
            except Exception:
                step = -1
            items.append((step, checkpoint.name))
    return [name for _, name in sorted(items)]

print('Repo dir:', REPO_DIR)
print('Git ref :', GITHUB_REF)
print('Run name:', RUN_NAME)
print('Checkpoint dir exists:', CHECKPOINT_DIR.exists())
print('Latest checkpoints:', sorted_checkpoints(CHECKPOINT_DIR)[-10:])
print('Adapter dir exists:', ADAPTER_DIR.exists())
print('Adapter files:', sorted([p.name for p in ADAPTER_DIR.iterdir()]) if ADAPTER_DIR.exists() else [])
assert (ADAPTER_DIR / 'adapter_config.json').exists(), f'Missing adapter_config.json: {ADAPTER_DIR}'
assert any((ADAPTER_DIR / name).exists() for name in ['adapter_model.safetensors', 'adapter_model.bin']), f'Missing adapter weights: {ADAPTER_DIR}'


In [ ]:
# Cell 2: Ensure the deterministic v5.2 E2B dataset exists in the cloned repo.
import hashlib

DATASET_PATH = Path(os.environ.get('DATASET_PATH', str(REPO_DIR / 'finetune' / 'data' / 'gemmafit_v5_2_e2b_evidence_router.json')))
DATASET_GENERATOR = REPO_DIR / 'finetune' / 'data' / 'generate_v5_e2b_evidence_router.py'

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not DATASET_PATH.exists():
    print('Dataset missing; regenerating deterministic v5.2 tool-contract dataset.')
    cmd = [
        sys.executable,
        str(DATASET_GENERATOR),
        '--train-count', '50000',
        '--validation-count', '6000',
        '--out', str(DATASET_PATH),
        '--validate',
        '--hard-cases',
        '--zh-tw-ratio', '0.45',
        '--schema-fuzz-ratio', '0.25',
        '--tool-contract-v2',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)

dataset_payload = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
print('Dataset:', DATASET_PATH)
print('sha256 :', sha256_file(DATASET_PATH))
print('train rows:', len(dataset_payload.get('train', [])))
print('validation splits:', {k: len(v) for k, v in dataset_payload.get('validation', {}).items()})


In [ ]:
# Cell 3: Load base model + text-LoRA adapter for direct generation eval.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

LOAD_IN_4BIT = os.environ.get('LOAD_IN_4BIT', '1') == '1'
DTYPE_NAME = os.environ.get('GEN_EVAL_DTYPE', 'float16')
DTYPE = {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}.get(DTYPE_NAME, torch.float16)
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), token=HF_TOKEN, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    'device_map': 'auto',
    'token': HF_TOKEN,
    'trust_remote_code': False,
}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=DTYPE,
    )
else:
    model_kwargs['dtype'] = DTYPE

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()

def lora_b_stats(model) -> dict:
    text_tensors = 0
    nonzero_tensors = 0
    nonzero_params = 0
    blocked_tensors = 0
    for name, param in model.named_parameters():
        if 'lora_B' not in name:
            continue
        is_text = 'language_model' in name or 'text_model' in name or 'decoder' in name
        is_blocked = 'vision_tower' in name or 'audio_tower' in name
        nonzero = int((param.detach() != 0).sum().item())
        if is_blocked:
            blocked_tensors += 1
        if is_text:
            text_tensors += 1
            nonzero_tensors += int(nonzero > 0)
            nonzero_params += nonzero
    return {
        'text_lora_B_tensors': text_tensors,
        'text_lora_B_nonzero_tensors': nonzero_tensors,
        'text_lora_B_nonzero_params': nonzero_params,
        'blocked_lora_B_tensors': blocked_tensors,
    }

print('Model loaded. load_in_4bit:', LOAD_IN_4BIT, 'dtype:', DTYPE_NAME)
print('LoRA stats:', json.dumps(lora_b_stats(model), indent=2))


In [ ]:
# Cell 4: Run real generation eval on held-out validation prompts.
# Start with GEN_EVAL_LIMIT=100. Raise to 300 or 1000 only after the 100-row gate looks sane.
import copy
import re
import time

from eval_v5_e2b_model_generation import (
    balanced_sample,
    expected_output,
    generated_eval_row,
    iter_selected_rows,
    parse_csv,
    prompt_messages,
    render_prompt,
    row_type_summary,
)
from eval_v5_e2b_evidence_router import check_gates, evaluate_rows, extract_json_object, subset_report

LIMIT = int(os.environ.get('GEN_EVAL_LIMIT', '100'))
SEED = int(os.environ.get('GEN_EVAL_SEED', '42'))
SPLITS = os.environ.get('GEN_EVAL_SPLITS', 'all')
ROW_TYPES = os.environ.get('GEN_EVAL_ROW_TYPES', 'all')
MAX_NEW_TOKENS = int(os.environ.get('MAX_NEW_TOKENS', '768'))
MAX_PROMPT_TOKENS = int(os.environ.get('MAX_PROMPT_TOKENS', '3072'))

RUN_NAME_SAFE = re.sub(r'[^A-Za-z0-9_.-]+', '_', RUN_NAME)
OUTPUT_JSON = Path(os.environ.get('GEN_EVAL_OUTPUT', str(EVAL_DIR / f'{RUN_NAME_SAFE}_eval_{LIMIT}.json')))
PREDICTIONS_JSONL = Path(os.environ.get('GEN_EVAL_PREDICTIONS', str(EVAL_DIR / f'{RUN_NAME_SAFE}_predictions_{LIMIT}.jsonl')))
ALLOW_PROMPT_TRUNCATION = os.environ.get('ALLOW_PROMPT_TRUNCATION', '0') == '1'

dataset = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
rows = iter_selected_rows(
    dataset,
    splits=parse_csv(SPLITS),
    row_types=parse_csv(ROW_TYPES),
    include_train=False,
)
rows = balanced_sample(rows, LIMIT, SEED)
assert rows, 'No rows selected for generation eval.'

device = next((param.device for param in model.parameters() if param.device.type != 'meta'), torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
generated_rows = []
predictions = []
parse_success = 0
started = time.time()

for index, row in enumerate(rows, start=1):
    prompt = render_prompt(row, tokenizer)
    encoded = tokenizer(prompt, return_tensors='pt', truncation=False)
    prompt_token_count = encoded['input_ids'].shape[-1]
    if prompt_token_count > MAX_PROMPT_TOKENS:
        if not ALLOW_PROMPT_TRUNCATION:
            raise RuntimeError(
                f'Prompt has {prompt_token_count} tokens, above MAX_PROMPT_TOKENS={MAX_PROMPT_TOKENS}; '
                'raise MAX_PROMPT_TOKENS or fix dataset prompt length instead of silently truncating eval input.'
            )
        encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_PROMPT_TOKENS)
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0][encoded['input_ids'].shape[-1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)

    parsed = None
    parse_error = ''
    try:
        parsed = extract_json_object(raw)
        parse_success += 1
    except Exception as exc:
        parse_error = str(exc)

    generated_rows.append(generated_eval_row(row, parsed, raw))
    predictions.append({
        'index': index - 1,
        'row_type': row.get('row_type', ''),
        'expected_function': row.get('expected_function', ''),
        'expected': expected_output(row),
        'parsed': parsed,
        'parse_error': parse_error,
        'raw_tail': raw[-1200:],
    })
    if index == 1 or index % 10 == 0 or index == len(rows):
        print(f'generated {index}/{len(rows)}')

report = evaluate_rows(generated_rows)
validations = report.pop('validations')
report['gate_failures'] = check_gates(report['summary'])
generation = {
    'base_model': BASE_MODEL_ID,
    'adapter': str(ADAPTER_DIR),
    'dataset': str(DATASET_PATH),
    'selected_rows': len(rows),
    'limit': LIMIT,
    'splits': SPLITS,
    'row_types': ROW_TYPES,
    'json_parse_rate': parse_success / len(rows),
    'elapsed_sec': round(time.time() - started, 3),
    'max_new_tokens': MAX_NEW_TOKENS,
    'max_prompt_tokens': MAX_PROMPT_TOKENS,
    'load_in_4bit': LOAD_IN_4BIT,
    'dtype': DTYPE_NAME,
}
report['generation'] = generation
if generation['json_parse_rate'] < 0.98:
    report['gate_failures'].append('json_parse_rate<0.98')
report['row_type_summary'] = row_type_summary(rows, validations)
report['samples'] = predictions[:12]
report['failure_samples'] = [
    {**predictions[item['index']], 'issues': item['issues']}
    for item in validations
    if item.get('issues')
][:100]
report['refusal_subset'] = subset_report(validations, {'unsupported', 'unsupported_zh_tw'})
report['adversarial_subset'] = subset_report(validations, {'adversarial'})

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
with PREDICTIONS_JSONL.open('w', encoding='utf-8') as f:
    for item in predictions:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(json.dumps({'summary': report['summary'], 'generation': generation, 'gate_failures': report['gate_failures']}, indent=2, ensure_ascii=False))
print('REPORT:', OUTPUT_JSON)
print('PREDICTIONS:', PREDICTIONS_JSONL)


In [ ]:
# Cell 5: Inspect failure samples.
report = json.loads(OUTPUT_JSON.read_text(encoding='utf-8'))
print(json.dumps({
    'summary': report.get('summary'),
    'generation': report.get('generation'),
    'gate_failures': report.get('gate_failures'),
    'row_type_summary': report.get('row_type_summary'),
}, indent=2, ensure_ascii=False))

failures = report.get('failure_samples', [])
print('Failure sample count:', len(failures))
for item in failures[:5]:
    print(json.dumps(item, indent=2, ensure_ascii=False)[:3000])


In [ ]:
# Cell 6: Failure analysis report.
# Run this after Cell 4. It groups the failures so the next fix is data/prompt driven, not guesswork.
from collections import Counter
import re

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

if 'OUTPUT_JSON' not in globals():
    fallback_limit = int(os.environ.get('GEN_EVAL_LIMIT', '100'))
    fallback_eval_dir = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train')) / 'trained_outputs' / 'evals'
    fallback_run_name = os.environ.get('RUN_NAME', 'text_lora_v1')
    fallback_run_name_safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', fallback_run_name)
    OUTPUT_JSON = Path(os.environ.get('GEN_EVAL_OUTPUT', str(fallback_eval_dir / f'{fallback_run_name_safe}_eval_{fallback_limit}.json')))
    PREDICTIONS_JSONL = Path(os.environ.get('GEN_EVAL_PREDICTIONS', str(fallback_eval_dir / f'{fallback_run_name_safe}_predictions_{fallback_limit}.jsonl')))

report = json.loads(OUTPUT_JSON.read_text(encoding='utf-8'))
predictions = load_jsonl(PREDICTIONS_JSONL)
failures = report.get('failure_samples', [])

issue_counts = Counter()
missing_args = Counter()
for item in failures:
    for issue in item.get('issues', []):
        issue_counts[issue] += 1
        if issue.startswith('missing_args:'):
            for arg in issue.split(':', 1)[1].split(','):
                missing_args[arg] += 1

def normalize_parse_error(value: str) -> str:
    if not value:
        return 'none'
    value = re.sub(r'line \d+ column \d+.*', 'json_decode_error', value)
    value = re.sub(r'char \d+', 'char_n', value)
    return value[:160]

parse_error_counts = Counter(normalize_parse_error(item.get('parse_error', '')) for item in predictions if item.get('parse_error'))

def parsed_function(item: dict) -> str:
    parsed = item.get('parsed')
    if isinstance(parsed, dict):
        return str(parsed.get('function') or '<missing_function>')
    if item.get('parse_error'):
        return '<parse_failed>'
    return '<missing_parsed>'

confusion = Counter((str(item.get('expected_function') or '<missing_expected>'), parsed_function(item)) for item in predictions)
row_type_failures = Counter(item.get('row_type', 'unknown') for item in failures)
forbidden_samples = [item for item in failures if 'forbidden_claim' in item.get('issues', [])][:10]
parse_fail_samples = [item for item in predictions if item.get('parse_error')][:10]
schema_samples = [item for item in failures if any(str(issue).startswith('missing_args') for issue in item.get('issues', []))][:10]

analysis = {
    'source_report': str(OUTPUT_JSON),
    'source_predictions': str(PREDICTIONS_JSONL),
    'summary': report.get('summary', {}),
    'gate_failures': report.get('gate_failures', []),
    'issue_counts': dict(issue_counts.most_common()),
    'missing_args': dict(missing_args.most_common()),
    'parse_error_counts': dict(parse_error_counts.most_common()),
    'function_confusion_top': [
        {'expected': expected, 'actual': actual, 'count': count}
        for (expected, actual), count in confusion.most_common(50)
    ],
    'row_type_failure_counts': dict(row_type_failures.most_common()),
    'row_type_summary': report.get('row_type_summary', {}),
    'forbidden_samples': forbidden_samples,
    'parse_fail_samples': parse_fail_samples,
    'schema_samples': schema_samples,
}

ANALYSIS_JSON = OUTPUT_JSON.with_name(OUTPUT_JSON.stem + '_failure_analysis.json')
ANALYSIS_JSON.write_text(json.dumps(analysis, indent=2, ensure_ascii=False), encoding='utf-8')

print('FAILURE_ANALYSIS:', ANALYSIS_JSON)
print(json.dumps({
    'summary': analysis['summary'],
    'gate_failures': analysis['gate_failures'],
    'issue_counts': dict(issue_counts.most_common(25)),
    'missing_args': dict(missing_args.most_common(25)),
    'parse_error_counts': dict(parse_error_counts.most_common(10)),
    'function_confusion_top': analysis['function_confusion_top'][:20],
    'row_type_failure_counts': analysis['row_type_failure_counts'],
}, indent=2, ensure_ascii=False))

print('\nParse failure samples:')
for item in parse_fail_samples[:3]:
    print(json.dumps(item, indent=2, ensure_ascii=False)[:2500])

print('\nForbidden claim samples:')
for item in forbidden_samples[:3]:
    print(json.dumps(item, indent=2, ensure_ascii=False)[:2500])


In [ ]:
# Cell 7: Optional fp16 merge for later export. Default is skipped.
# Set MERGE_FP16=1 before running this cell when you need a merged HF model directory.
MERGE_FP16 = os.environ.get('MERGE_FP16', '0') == '1'
if not MERGE_FP16:
    print('Skipping fp16 merge. Set MERGE_FP16=1 to enable.')
else:
    import gc
    try:
        del model
        del base_model
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    merge_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=torch.float16,
        device_map='auto',
        token=HF_TOKEN,
        trust_remote_code=False,
    )
    merge_model = PeftModel.from_pretrained(merge_base, str(ADAPTER_DIR))
    merged = merge_model.merge_and_unload()
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    tokenizer.save_pretrained(str(MERGED_DIR))
    print('MERGED_READY:', MERGED_DIR)
    print('files:', sorted(p.name for p in MERGED_DIR.iterdir()))
